# Rastreador de Preços — GPUs e PS5

Este notebook coleta, armazena e analisa o histórico de preços de placas de
vídeo e consoles:

- **AMD RX 9070 XT 16GB** — ASRock Challenger
- **NVIDIA RTX 5070 Ti 16GB** — MSI Shadow 3X OC
- **NVIDIA RTX 5070 12GB** — menor preço por loja (MSI Ventus 2X OC na
  Kabum, Palit White OC na Terabyte — ver nota abaixo)
- **Sony PlayStation 5 Edição Digital 825GB** — menor preço entre lojas via
  Buscapé
- **Sony PlayStation 5 Slim 1TB** — menor preço entre lojas via Buscapé

Objetivo: entender o comportamento do mercado (quando o preço sobe, quando
desce, sazonalidade) para identificar um bom momento de compra.

**Como usar:**
1. Rode a célula de coleta periodicamente (ex.: uma vez por dia) — cada
   execução adiciona uma linha nova em `data/historico_precos.csv`, sem
   apagar o que já foi coletado.
2. Rode as células de análise sempre que quiser ver o estado atual.
3. Quanto mais tempo você deixar isso rodando, mais confiável fica a análise
   de média móvel e sazonalidade — com poucos pontos, os gráficos vão
   aparecer, mas os padrões ainda não serão estatisticamente sólidos.

**Sobre a coleta:** os preços das GPUs são obtidos via scraping direto de
Kabum e Terabyte (dados estruturados JSON-LD) e via busca no Buscapé
(comparador de preços — os resultados de busca vêm embutidos como JSON na
própria página, e o menor preço é calculado entre os anúncios que batem com
o modelo procurado). O PS5 também usa o Buscapé, mas via página de produto
(que já expõe o menor preço agregado em JSON-LD diretamente, sem precisar
de busca). Até meados de setembro/2026 quem alimentava essa "busca" das
GPUs era o Promotech (outro comparador de preços); ele foi substituído pelo
Buscapé porque passou a bloquear toda requisição automatizada (mesmo a
home) com um desafio "Vercel Security Checkpoint" (HTTP 429), na mesma
linha do bloqueio Cloudflare da Pichau. O link do
[Promotech](https://promotech.app.br) fica no notebook só como referência
para conferência manual. A Pichau não está entre as lojas ativas pelo mesmo
motivo (bloqueio deliberado de bots, não link quebrado). Se algum dos
outros sites mudar de layout, o scraper daquela loja pode parar de retornar
preço — isso é normal nesse tipo de ferramenta, veja a seção "Solução de
problemas" no fim do notebook.

**Sobre a RTX 5070 12GB:** diferente das outras duas linhas de GPU, aqui
cada loja aponta para um modelo/fabricante diferente (não existe uma SKU
única sendo comparada entre lojas) — o que é acompanhado é o menor preço de
uma RTX 5070 12GB disponível em cada loja, não um cooler/fabricante fixo.
Se preferir travar numa SKU específica por loja, edite `PRODUTOS_MODELOS`
em `config.py`.

**Sobre o PS5:** Edição Digital e Slim 1TB são acompanhados como modelos
separados (preços bem diferentes, não fazem sentido combinados). Cada um
usa uma única URL do Buscapé, então "loja" aqui sempre aparece como
"Buscapé (menor preço)" — é o comparador que já faz o trabalho de achar o
menor preço entre os vendedores que ele lista para aquele anúncio.


In [1]:
import sys
from pathlib import Path
from datetime import datetime

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

sys.path.append(str(Path.cwd()))
import config
import scrapers
import cotacao_dolar

CSV_HISTORICO = Path(config.CSV_HISTORICO)
CSV_DOLAR = Path(config.CSV_DOLAR)
print("Ambiente carregado.")

Ambiente carregado.


## 1. Coletar preços agora

Rode esta célula sempre que quiser adicionar um novo ponto ao histórico. Ela não apaga nada — só acrescenta linhas novas ao CSV.

In [2]:
# debug=True salva o HTML bruto em data/debug_html/ quando um scraper falha
# em achar o preço — útil para ajustar a extração se um site mudar o layout.
novos_registros = scrapers.coletar_todos(config.PRODUTOS_MODELOS, debug=True)

if novos_registros:
    df_novos = pd.DataFrame(novos_registros)
    cabecalho = not CSV_HISTORICO.exists()
    df_novos.to_csv(CSV_HISTORICO, mode="a", header=cabecalho, index=False)
    print(f"\n{len(novos_registros)} novo(s) registro(s) salvo(s) em {CSV_HISTORICO}")
else:
    print("Nenhum preço novo coletado nesta execução (veja mensagens de erro acima).")


[OK] RX 9070 XT 16GB — Kabum: R$ 5899.99
[OK] RX 9070 XT 16GB — Terabyte: R$ 5399.90
[OK] RX 9070 XT 16GB — Promotech (menor preço): R$ 4899.89
[OK] RTX 5070 Ti 16GB — Kabum: R$ 7649.91
[OK] RTX 5070 Ti 16GB — Terabyte: R$ 8549.91
[OK] RTX 5070 Ti 16GB — Promotech (menor preço): R$ 7739.14
[OK] RTX 5070 12GB — Kabum: R$ 5599.99
[OK] RTX 5070 12GB — Terabyte: R$ 5199.90
[OK] RTX 5070 12GB — Promotech (menor preço): R$ 4999.99
[OK] PS5 Edição Digital 825GB — Buscapé (menor preço): R$ 4299.00
[OK] PS5 Slim 1TB — Buscapé (menor preço): R$ 4599.90

11 novo(s) registro(s) salvo(s) em data\historico_precos.csv


## 2. Cotação do dólar (USD/BRL)

GPU no Brasil é majoritariamente importada, então o câmbio costuma explicar parte da variação de preço. Isso já traz histórico de até um ano, então não precisamos esperar semanas para ter esse dado.

In [3]:
try:
    df_dolar = cotacao_dolar.obter_historico_dolar(dias=365)
    df_dolar.to_csv(CSV_DOLAR, index=False)
    print(f"{len(df_dolar)} cotações diárias carregadas.")
    display(df_dolar.tail())
except Exception as e:
    print(f"Não foi possível obter a cotação do dólar agora: {e}")
    print("Sem problema — rode esta célula de novo mais tarde. A Seção 8 (preço x dólar) "
          "vai avisar e pular automaticamente até que exista data/cotacao_dolar.csv.")


360 cotações diárias carregadas.


,data,cotacao_venda
355,2026-09-04,5.1249
356,2026-09-07,5.1259
357,2026-09-08,5.0867
358,2026-09-09,5.1048
359,2026-09-10,5.1033


## 3. Carregar o histórico de preços

In [4]:
df = pd.read_csv(CSV_HISTORICO, parse_dates=["data_hora"])
df["data"] = df["data_hora"].dt.date
df = df.sort_values("data_hora").reset_index(drop=True)

print(f"{len(df)} registros no total, de {df['data_hora'].min()} até {df['data_hora'].max()}")
df.tail(10)


306 registros no total, de 2026-08-13 00:00:00 até 2026-09-11 21:14:14


,data_hora,loja,modelo,preco,titulo_produto,url,data
296,2026-09-11 21:13:33,Terabyte,RX 9070 XT 16GB,5399.90,Placa de Vídeo ASRock RX 9070 XT Challenger 16...,https://www.terabyteshop.com.br/produto/38584/...,2026-09-11
297,2026-09-11 21:13:37,Promotech (menor preço),RX 9070 XT 16GB,4899.89,"ASRock RX 9070 XT Challenger R$ 4.899,89 ou R$...",https://promotech.app.br/busca?q=Rx+9070+XT,2026-09-11
298,2026-09-11 21:13:41,Kabum,RTX 5070 Ti 16GB,7649.91,Placa De Video Msi Geforce RTX 5070 Ti Shadow ...,https://www.kabum.com.br/produto/779497/placa-...,2026-09-11
299,2026-09-11 21:13:47,Terabyte,RTX 5070 Ti 16GB,8549.91,GPU MSI RTX 5070 Ti Shadow 3X OC 16GB GDDR7 | ...,https://www.terabyteshop.com.br/produto/35475/...,2026-09-11
300,2026-09-11 21:13:52,Promotech (menor preço),RTX 5070 Ti 16GB,7739.14,"MSI RTX 5070 Ti Shadow 3X OC R$ 7.739,14 ou R$...",https://promotech.app.br/busca?q=RTX+5070+TI,2026-09-11
301,2026-09-11 21:13:57,Kabum,RTX 5070 12GB,5599.99,Placa de Vídeo MSI GeForce RTX 5070 12G VENTUS...,https://www.kabum.com.br/produto/725587/placa-...,2026-09-11
302,2026-09-11 21:14:01,Terabyte,RTX 5070 12GB,5199.90,Placa de Vídeo Palit RTX 5070 White OC 12GB GD...,https://www.terabyteshop.com.br/produto/40135/...,2026-09-11
303,2026-09-11 21:14:07,Promotech (menor preço),RTX 5070 12GB,4999.99,"PNY RTX 5070 OC R$ 4.999,99 ou R$ 5.882,40 par...",https://promotech.app.br/busca?q=RTX+5070,2026-09-11
304,2026-09-11 21:14:10,Buscapé (menor preço),PS5 Edição Digital 825GB,4299.00,PlayStation 5 Edição Digital 825GB,https://www.buscape.com.br/console-de-video-ga...,2026-09-11
305,2026-09-11 21:14:14,Buscapé (menor preço),PS5 Slim 1TB,4599.90,Playstation 5 Slim Edição Digital 1TB,https://www.buscape.com.br/console-de-video-ga...,2026-09-11


## 4. Série de preços por modelo (bruto + média móvel)

A média móvel só aparece quando já existem pontos suficientes — com poucos dias de coleta, o gráfico mostra só os preços brutos.

In [5]:
JANELA_MEDIA_MOVEL = 7  # dias

cores_plotly = {
    "RX 9070 XT 16GB": "#d62728",
    "RTX 5070 Ti 16GB": "#2ca02c",
    "RTX 5070 12GB": "#1f77b4",
    "PS5 Edição Digital 825GB": "#9467bd",
    "PS5 Slim 1TB": "#ff7f0e",
}

fig = go.Figure()

for modelo, grupo in df.groupby("modelo"):
    cor = cores_plotly.get(modelo)
    diario = grupo.groupby("data")["preco"].mean().reset_index()
    diario["data"] = pd.to_datetime(diario["data"])

    fig.add_trace(
        go.Scatter(
            x=diario["data"],
            y=diario["preco"],
            name=f"{modelo} (bruto)",
            mode="lines+markers",
            opacity=0.5,
            line=dict(color=cor, width=1.5),
            marker=dict(size=4),
            hovertemplate="%{x|%d/%m/%Y}<br>R$ %{y:,.2f}<extra>" + f"{modelo} (bruto)" + "</extra>",
        )
    )

    if len(diario) >= JANELA_MEDIA_MOVEL:
        diario["media_movel"] = diario["preco"].rolling(JANELA_MEDIA_MOVEL, min_periods=1).mean()
        fig.add_trace(
            go.Scatter(
                x=diario["data"],
                y=diario["media_movel"],
                name=f"{modelo} (média móvel {JANELA_MEDIA_MOVEL}d)",
                mode="lines",
                line=dict(color=cor, width=2.5, dash="dash"),
                hovertemplate="%{x|%d/%m/%Y}<br>R$ %{y:,.2f}<extra>"
                + f"{modelo} (média móvel {JANELA_MEDIA_MOVEL}d)"
                + "</extra>",
            )
        )

fig.update_layout(
    title="Histórico de preços por modelo",
    yaxis_title="Preço (R$)",
    template="plotly_white",
    hovermode="x unified",
    height=550,
)
fig.show()

## 6. Estatísticas resumo por modelo

In [6]:
linhas_resumo = []
for modelo, grupo in df.groupby("modelo"):
    atual = grupo.sort_values("data_hora")["preco"].iloc[-1]
    linhas_resumo.append({
        "modelo": modelo,
        "preco_atual": atual,
        "preco_minimo": grupo["preco"].min(),
        "preco_maximo": grupo["preco"].max(),
        "preco_medio": grupo["preco"].mean(),
        "percentil_atual": (grupo["preco"] <= atual).mean() * 100,
        "n_coletas": len(grupo),
    })

resumo = pd.DataFrame(linhas_resumo).set_index("modelo").round(2)
resumo


,preco_atual,preco_minimo,preco_maximo,preco_medio,percentil_atual,n_coletas
modelo,,,,,,
PS5 Edição Digital 825GB,4299.00,4199.90,4299.00,4267.47,100.00,20
PS5 Slim 1TB,4599.90,3999.99,4599.90,4319.86,100.00,20
RTX 5070 12GB,4999.99,4599.99,6000.00,4992.09,83.33,66
RTX 5070 Ti 16GB,7739.14,7083.08,8649.00,7813.47,67.00,100
RX 9070 XT 16GB,4899.89,4699.99,6999.99,5188.14,35.00,100


## 7. Sazonalidade

Padrão de preço por dia da semana e por dia do mês. **Esses gráficos só ficam confiáveis depois de algumas semanas/meses de coleta acumulada** — com poucos dias de dado, o padrão que aparece aqui é ruído, não sazonalidade real.

In [7]:
MINIMO_DIAS_PARA_SAZONALIDADE = 21

if df["data"].nunique() < MINIMO_DIAS_PARA_SAZONALIDADE:
    print(
        f"Ainda há apenas {df['data'].nunique()} dia(s) distinto(s) de coleta. "
        f"Recomendo esperar pelo menos {MINIMO_DIAS_PARA_SAZONALIDADE} dias de "
        "coleta acumulada antes de tirar conclusões de sazonalidade."
    )
else:
    df["dia_semana"] = pd.to_datetime(df["data"]).dt.day_name()
    ordem_dias = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

    cores_plotly = {
        "RX 9070 XT 16GB": "#d62728",
        "RTX 5070 Ti 16GB": "#2ca02c",
        "RTX 5070 12GB": "#1f77b4",
        "PS5 Edição Digital 825GB": "#9467bd",
        "PS5 Slim 1TB": "#ff7f0e",
    }

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Preço médio por dia da semana", "Preço médio por dia do mês"),
    )

    for modelo, grupo in df.groupby("modelo"):
        cor = cores_plotly.get(modelo)

        media_dia_semana = grupo.groupby("dia_semana")["preco"].mean().reindex(ordem_dias)
        fig.add_trace(
            go.Scatter(
                x=media_dia_semana.index,
                y=media_dia_semana.values,
                name=modelo,
                mode="lines+markers",
                line=dict(color=cor, width=2),
                marker=dict(size=6),
                legendgroup=modelo,
                hovertemplate="%{x}<br>R$ %{y:,.2f}<extra>" + modelo + "</extra>",
            ),
            row=1, col=1,
        )

        grupo_mes = grupo.copy()
        grupo_mes["dia_mes"] = pd.to_datetime(grupo_mes["data"]).dt.day
        media_dia_mes = grupo_mes.groupby("dia_mes")["preco"].mean()
        fig.add_trace(
            go.Scatter(
                x=media_dia_mes.index,
                y=media_dia_mes.values,
                name=modelo,
                mode="lines+markers",
                line=dict(color=cor, width=2),
                marker=dict(size=6),
                legendgroup=modelo,
                showlegend=False,
                hovertemplate="Dia %{x}<br>R$ %{y:,.2f}<extra>" + modelo + "</extra>",
            ),
            row=1, col=2,
        )

    fig.update_xaxes(tickangle=45, row=1, col=1)
    fig.update_xaxes(title_text="Dia do mês", row=1, col=2)
    fig.update_yaxes(title_text="Preço médio (R$)", row=1, col=1)

    fig.update_layout(
        title="Sazonalidade de preços",
        template="plotly_white",
        height=500,
        legend=dict(orientation="h", yanchor="bottom", y=1.15, xanchor="left", x=0),
        margin=dict(t=140),
    )
    fig.show()

## 8. Preço x cotação do dólar

Sobrepõe o preço de cada modelo à cotação do dólar no mesmo período, para ajudar a distinguir alta de preço por escassez/demanda de alta puxada pelo câmbio.

In [8]:
if not CSV_DOLAR.exists():
    print("Ainda não há data/cotacao_dolar.csv — rode a célula da Seção 2 com sucesso primeiro.")
else:
    df_dolar_local = pd.read_csv(CSV_DOLAR, parse_dates=["data"])
    df_dolar_local["data"] = df_dolar_local["data"].dt.date

    cores_plotly = {
        "RX 9070 XT 16GB": "#d62728",
        "RTX 5070 Ti 16GB": "#2ca02c",
        "RTX 5070 12GB": "#1f77b4",
        "PS5 Edição Digital 825GB": "#9467bd",
        "PS5 Slim 1TB": "#ff7f0e",
    }

    fig = make_subplots(specs=[[{"secondary_y": True}]])

    for modelo, grupo in df.groupby("modelo"):
        diario = grupo.groupby("data")["preco"].mean().reset_index()
        diario["data"] = pd.to_datetime(diario["data"])
        fig.add_trace(
            go.Scatter(
                x=diario["data"],
                y=diario["preco"],
                name=modelo,
                mode="lines+markers",
                line=dict(color=cores_plotly.get(modelo), width=2),
                marker=dict(size=5),
                hovertemplate="%{x|%d/%m/%Y}<br>R$ %{y:,.2f}<extra>" + modelo + "</extra>",
            ),
            secondary_y=False,
        )

    dolar_no_periodo = df_dolar_local[df_dolar_local["data"] >= df["data"].min()]
    fig.add_trace(
        go.Scatter(
            x=pd.to_datetime(dolar_no_periodo["data"]),
            y=dolar_no_periodo["cotacao_venda"],
            name="USD/BRL",
            mode="lines",
            line=dict(color="#7f7f7f", width=2, dash="dot"),
            hovertemplate="%{x|%d/%m/%Y}<br>R$ %{y:.4f}<extra>USD/BRL</extra>",
        ),
        secondary_y=True,
    )

    fig.update_layout(
        title="Preço das GPUs e do PS5 vs. cotação do dólar",
        template="plotly_white",
        hovermode="x unified",
        height=550,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        margin=dict(t=90),
    )
    fig.update_yaxes(title_text="Preço (R$)", secondary_y=False)
    fig.update_yaxes(title_text="Cotação USD/BRL (R$)", secondary_y=True, showgrid=False)
    fig.show()

## 9. Sinal de possível bom momento de compra

Compara o preço mais recente de cada modelo com a média móvel — quando o preço atual está bem abaixo da média recente, pode ser um indício de boa oportunidade. **Isso é só um indicador estatístico simples, não uma recomendação financeira.**

In [9]:
LIMIAR_ALERTA_PCT = 5  # considerar "abaixo da média" a partir de X%

for modelo, grupo in df.groupby("modelo"):
    diario_medio = grupo.groupby("data")["preco"].mean().reset_index()
    diario_minimo = grupo.groupby("data")["preco"].min().reset_index()

    if len(diario_medio) < 3:
        print(f"{modelo}: poucos pontos ainda ({len(diario_medio)}) para calcular um sinal confiável.\n")
        continue

    media_movel = diario_medio["preco"].rolling(min(JANELA_MEDIA_MOVEL, len(diario_medio)), min_periods=1).mean().iloc[-1]
    preco_atual = diario_minimo["preco"].iloc[-1]
    variacao_pct = (preco_atual - media_movel) / media_movel * 100

    sinal = "📉 possível bom momento" if variacao_pct <= -LIMIAR_ALERTA_PCT else (
        "📈 acima da média recente" if variacao_pct >= LIMIAR_ALERTA_PCT else "➡️ dentro da média recente"
    )

    print(f"{modelo}")
    print(f"  Menor preço atual: R$ {preco_atual:,.2f}")
    print(f"  Média móvel recente: R$ {media_movel:,.2f}")
    print(f"  Variação: {variacao_pct:+.1f}%  →  {sinal}\n")


PS5 Edição Digital 825GB
  Menor preço atual: R$ 4,299.00
  Média móvel recente: R$ 4,278.00
  Variação: +0.5%  →  ➡️ dentro da média recente

PS5 Slim 1TB
  Menor preço atual: R$ 4,599.90
  Média móvel recente: R$ 4,356.93
  Variação: +5.6%  →  📈 acima da média recente

RTX 5070 12GB
  Menor preço atual: R$ 4,999.99
  Média móvel recente: R$ 5,098.07
  Variação: -1.9%  →  ➡️ dentro da média recente

RTX 5070 Ti 16GB
  Menor preço atual: R$ 7,649.91
  Média móvel recente: R$ 7,979.65
  Variação: -4.1%  →  ➡️ dentro da média recente

RX 9070 XT 16GB
  Menor preço atual: R$ 4,899.89
  Média móvel recente: R$ 5,366.60
  Variação: -8.7%  →  📉 possível bom momento



## 10. Variação do dólar no período de coleta

Resume quanto o USD/BRL andou entre o primeiro e o último dia cobertos pelo seu histórico de preços (não os 365 dias completos da Seção 2, só o recorte que se sobrepõe à sua coleta).

In [10]:
if not CSV_DOLAR.exists():
    print("Ainda não há data/cotacao_dolar.csv — rode a célula da Seção 2 com sucesso primeiro.")
else:
    df_dolar_periodo = pd.read_csv(CSV_DOLAR, parse_dates=["data"])
    df_dolar_periodo["data"] = df_dolar_periodo["data"].dt.date

    inicio, fim = df["data"].min(), df["data"].max()
    df_dolar_periodo = df_dolar_periodo[
        (df_dolar_periodo["data"] >= inicio) & (df_dolar_periodo["data"] <= fim)
    ].sort_values("data")

    if len(df_dolar_periodo) < 2:
        print(
            f"Só há {len(df_dolar_periodo)} cotação(ões) do dólar no período coletado "
            f"({inicio} a {fim}) — sem pontos suficientes para calcular variação."
        )
    else:
        cotacao_inicial = df_dolar_periodo["cotacao_venda"].iloc[0]
        cotacao_final = df_dolar_periodo["cotacao_venda"].iloc[-1]
        cotacao_minima = df_dolar_periodo["cotacao_venda"].min()
        cotacao_maxima = df_dolar_periodo["cotacao_venda"].max()
        variacao_pct = (cotacao_final - cotacao_inicial) / cotacao_inicial * 100

        print(f"Período de coleta: {inicio} a {fim}")
        print(f"Dólar em {inicio}: R$ {cotacao_inicial:,.4f}")
        print(f"Dólar em {fim}: R$ {cotacao_final:,.4f}")
        print(f"Variação no período: {variacao_pct:+.2f}%")
        print(f"Mínima: R$ {cotacao_minima:,.4f}  |  Máxima: R$ {cotacao_maxima:,.4f}")

        fig = go.Figure(
            go.Scatter(
                x=pd.to_datetime(df_dolar_periodo["data"]),
                y=df_dolar_periodo["cotacao_venda"],
                mode="lines+markers",
                line=dict(color="#1f77b4", width=2),
                marker=dict(size=5),
                hovertemplate="%{x|%d/%m/%Y}<br>R$ %{y:.4f}<extra></extra>",
            )
        )
        fig.update_layout(
            title=f"Cotação USD/BRL no período de coleta ({inicio} a {fim})",
            yaxis_title="Cotação USD/BRL (R$)",
            template="plotly_white",
            hovermode="x unified",
            height=500,
        )
        fig.show()

Período de coleta: 2026-08-13 a 2026-09-11
Dólar em 2026-08-13: R$ 5.1795
Dólar em 2026-09-11: R$ 5.1033
Variação no período: -1.47%
Mínima: R$ 5.0867  |  Máxima: R$ 5.2225


## 11. Histórico completo do dólar (interativo)

Mesma ideia da Seção 10, mas sem recortar pelo período de coleta — usa todas as cotações que existem em `data/cotacao_dolar.csv` (até ~365 dias, dependendo de quando a Seção 2 foi rodada pela última vez). O gráfico aqui é interativo (Plotly): passe o mouse sobre a linha para ver a data e a cotação exata daquele ponto.

In [11]:
if not CSV_DOLAR.exists():
    print("Ainda não há data/cotacao_dolar.csv — rode a célula da Seção 2 com sucesso primeiro.")
else:
    df_dolar_completo = pd.read_csv(CSV_DOLAR, parse_dates=["data"]).sort_values("data")

    inicio, fim = df_dolar_completo["data"].min(), df_dolar_completo["data"].max()
    cotacao_inicial = df_dolar_completo["cotacao_venda"].iloc[0]
    cotacao_final = df_dolar_completo["cotacao_venda"].iloc[-1]
    cotacao_minima = df_dolar_completo["cotacao_venda"].min()
    cotacao_maxima = df_dolar_completo["cotacao_venda"].max()
    variacao_pct = (cotacao_final - cotacao_inicial) / cotacao_inicial * 100

    print(f"Período completo: {inicio.date()} a {fim.date()} ({len(df_dolar_completo)} cotações)")
    print(f"Dólar em {inicio.date()}: R$ {cotacao_inicial:,.4f}")
    print(f"Dólar em {fim.date()}: R$ {cotacao_final:,.4f}")
    print(f"Variação no período completo: {variacao_pct:+.2f}%")
    print(f"Mínima: R$ {cotacao_minima:,.4f}  |  Máxima: R$ {cotacao_maxima:,.4f}")

    fig = go.Figure(
        go.Scatter(
            x=df_dolar_completo["data"],
            y=df_dolar_completo["cotacao_venda"],
            mode="lines",
            line=dict(color="#1f77b4", width=2),
            hovertemplate="%{x|%d/%m/%Y}<br>R$ %{y:.4f}<extra></extra>",
        )
    )
    fig.update_layout(
        title=f"Cotação USD/BRL — histórico completo ({inicio.date()} a {fim.date()})",
        yaxis_title="Cotação USD/BRL (R$)",
        template="plotly_white",
        hovermode="x unified",
        height=500,
    )
    fig.show()


Período completo: 2025-07-18 a 2026-09-10 (360 cotações)
Dólar em 2025-07-18: R$ 5.5755
Dólar em 2026-09-10: R$ 5.1033
Variação no período completo: -8.47%
Mínima: R$ 4.9064  |  Máxima: R$ 5.5987


In [12]:
data_inicio = df_dolar_completo['data'].min()
data_fim = df_dolar_completo['data'].max()
dias_transcorridos = (data_fim - data_inicio).days
media_periodo = df_dolar_completo['cotacao_venda'].mean()
valor_hoje = df_dolar_completo['cotacao_venda'].iloc[-1]
valor_ontem = df_dolar_completo['cotacao_venda'].iloc[-2]
ultimos_5_dias = df_dolar_completo.tail(5)

tabela_cotacao = pd.DataFrame(
    {
'Hoje': valor_hoje,
'Var % anterior': ((valor_hoje - valor_ontem)/valor_ontem)*100,
'Var % ultimos 5 dias': ((valor_hoje - ultimos_5_dias['cotacao_venda'].mean())/ultimos_5_dias['cotacao_venda'].mean())*100,
"Media ult 5 d": ultimos_5_dias['cotacao_venda'].mean(),
'Var % periodo': ((valor_hoje - media_periodo)/media_periodo)*100,
'Média período analisado': media_periodo,
'Desvio padrão': [df_dolar_completo['cotacao_venda'].std()],
'Inicio obso': data_inicio,
'ultimo dia obs': data_fim,
'Dias transc': dias_transcorridos
}
)


colunas_numericas = tabela_cotacao.select_dtypes("number").columns
tabela_cotacao[colunas_numericas] = tabela_cotacao[colunas_numericas].round(3)
display(tabela_cotacao)


,Hoje,Var % anterior,Var % ultimos 5 dias,Media ult 5 d,Var % periodo,Média período analisado,Desvio padrão,Inicio obso,ultimo dia obs,Dias transc
0,5.103,-0.029,-0.114,5.109,-2.887,5.255,0.166,2025-07-18,2026-09-10,419


## Solução de problemas

**Um scraper não retornou preço nenhum:**
O site provavelmente mudou o layout. Rode a coleta com `debug=True` (já é o
padrão na célula da Seção 1) — isso salva o HTML bruto em
`data/debug_html/`. Abra o arquivo, procure o preço no código-fonte e ajuste
a função `_extrair_via_jsonld`/`_extrair_via_regex` (Kabum, Terabyte),
`_extrair_via_jsonld_buscape` (Buscapé/PS5) ou `_extrair_via_busca_buscape`
(Buscapé/busca de GPU) em `scrapers.py`.

**Quero adicionar outra loja ou trocar a SKU de um produto:**
Edite o dicionário `PRODUTOS_MODELOS` em `config.py`. Para uma loja com
página de produto própria (como Kabum/Terabyte), adicione uma chave
`<loja>_url` e inclua essa chave em `LOJAS_ATIVAS` (scrapers.py) — o
scraper genérico (`coletar_preco`, JSON-LD + fallback por regex) deve
funcionar sem mudanças. Para um comparador de preços com página de busca
(como o Buscapé para as GPUs), o padrão é uma chave
`buscape_busca_url`/`buscape_busca_termo` (e opcionalmente
`buscape_busca_excluir`, uma tupla de palavras a evitar no nome do anúncio)
e a função `coletar_preco_buscape_busca`, que filtra os resultados de busca
por palavra e usa o menor preço entre os que baterem — ver
`_extrair_via_busca_buscape`. Para um comparador cuja própria página de
produto já expõe o preço agregado em JSON-LD (como o Buscapé para o PS5),
basta uma chave `buscape_url` — ver `coletar_preco_buscape`.

**Quero automatizar a coleta (sem rodar o notebook manualmente todo dia):**
No Windows, o Agendador de Tarefas pode rodar
`jupyter nbconvert --to notebook --execute analise_precos.ipynb --output analise_precos.ipynb`
uma vez por dia. No Linux/Mac, o mesmo comando funciona via `cron`.

**Sobre confiabilidade dos scrapers:**
Kabum e Terabyte tendem a manter dados estruturados (JSON-LD) nas páginas de
produto, o que ajuda a manter o scraper estável mesmo com pequenas mudanças
visuais no site. O Buscapé também usa dados estruturados na página de
produto (JSON-LD, offers.lowPrice) e um JSON embutido (`__NEXT_DATA__`) na
página de busca — ambos mais estáveis que casar texto solto em HTML.
Ainda assim, nenhum scraper é permanente — trate ajustes ocasionais como
parte normal da manutenção da ferramenta.

**Por que a Pichau não está na lista de lojas?**
O site responde com um desafio Cloudflare (HTTP 403, "Just a moment...")
para requisições automatizadas — não é um problema de seletor CSS ou de
link quebrado, é bloqueio deliberado de bots. Contornar isso de forma
confiável exigiria automatizar um navegador para resolver o desafio, o que
fere os termos de uso do site, então optamos por deixar a Pichau de fora da
coleta ativa.

**Por que o Promotech parou de ser coletado automaticamente?**
Até meados de setembro/2026 ele alimentava a coleta via página de busca. Desde
então, o site passou a responder com um "Vercel Security Checkpoint" (HTTP
429, header `X-Vercel-Mitigated: challenge`) em qualquer requisição
automatizada — inclusive na home, não só na busca. É a mesma categoria de
bloqueio da Pichau (anti-bot deliberado, não link quebrado nem mudança de
layout), então foi removido de `coletar_todos()` em `scrapers.py` e
substituído pela busca via Buscapé (`buscape_busca_url`/`_termo`, ver acima).
O link `promotech_url` continua em `config.py` só para conferência manual
num navegador de verdade.
